# Download DermaXplain Datasets from Google Drive

This notebook downloads the prepared dataset ZIP files from Google Drive using `gdown`, extracts them, and verifies the expected directory structure.

Expected final structure:

```text
data/
├── ham10000/
│   ├── images/
│   ├── masks/
│   └── HAM10000_metadata.csv
└── isic2018/
    ├── images/
    └── masks/
```

Before running this notebook, make sure both Google Drive ZIP files are shared as:

```text
Anyone with the link → Viewer
```


In [ ]:
from pathlib import Path
import sys
import subprocess
import zipfile

# Detect project root.
# If this notebook is run from src/, PROJECT_ROOT becomes the parent folder.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "src":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
ARCHIVE_DIR = PROJECT_ROOT / "data_archives"

DATA_DIR.mkdir(exist_ok=True)
ARCHIVE_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data dir:    ", DATA_DIR)
print("Archive dir: ", ARCHIVE_DIR)


## 1. Install `gdown` if needed

In [ ]:
try:
    import gdown
    print("gdown already installed")
except ImportError:
    print("Installing gdown...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gdown"])
    import gdown
    print("gdown installed")


## 2. Add Google Drive links

Replace the placeholder values below with the Google Drive share links for:

- `ham10000.zip`
- `isic2018.zip`


In [ ]:
HAM10000_URL = "PASTE_HAM10000_GOOGLE_DRIVE_LINK_HERE"
ISIC2018_URL = "PASTE_ISIC2018_GOOGLE_DRIVE_LINK_HERE"

HAM10000_ZIP = ARCHIVE_DIR / "ham10000.zip"
ISIC2018_ZIP = ARCHIVE_DIR / "isic2018.zip"

print("HAM10000 target:", HAM10000_ZIP)
print("ISIC2018 target:", ISIC2018_ZIP)


## 3. Download archives

In [ ]:
import gdown

def download_if_missing(url: str, output_path: Path) -> None:
    """Download a Google Drive file with gdown unless it already exists."""
    if output_path.exists():
        print(f"Already exists, skipping: {output_path}")
        return

    if "PASTE_" in url:
        raise ValueError(f"Please replace the placeholder URL for {output_path.name}")

    print(f"Downloading to: {output_path}")
    result = gdown.download(url, str(output_path), quiet=False, fuzzy=True)

    if result is None or not output_path.exists():
        raise RuntimeError(f"Download failed for {output_path.name}")

download_if_missing(HAM10000_URL, HAM10000_ZIP)
download_if_missing(ISIC2018_URL, ISIC2018_ZIP)


## 4. Check archive sizes

In [ ]:
def size_gb(path: Path) -> float:
    return path.stat().st_size / (1024 ** 3)

for archive in [HAM10000_ZIP, ISIC2018_ZIP]:
    if archive.exists():
        print(f"{archive.name}: {size_gb(archive):.2f} GB")
    else:
        print(f"Missing: {archive}")


## 5. Extract archives

The ZIPs were created from the project root, so they should contain paths such as:

```text
data/ham10000/...
data/isic2018/...
```

Therefore, they are extracted into `PROJECT_ROOT`.


In [ ]:
def extract_if_needed(zip_path: Path, expected_dir: Path) -> None:
    """Extract ZIP into PROJECT_ROOT unless the expected folder already exists."""
    if expected_dir.exists():
        print(f"Already extracted, skipping: {expected_dir}")
        return

    if not zip_path.exists():
        raise FileNotFoundError(f"Missing archive: {zip_path}")

    print(f"Extracting: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(PROJECT_ROOT)
    print(f"Extracted: {zip_path.name}")

extract_if_needed(HAM10000_ZIP, DATA_DIR / "ham10000" / "images")
extract_if_needed(ISIC2018_ZIP, DATA_DIR / "isic2018" / "images")


## 6. Verify expected structure

In [ ]:
required_paths = [
    DATA_DIR / "ham10000" / "images",
    DATA_DIR / "ham10000" / "masks",
    DATA_DIR / "isic2018" / "images",
    DATA_DIR / "isic2018" / "masks",
]

all_ok = True

for path in required_paths:
    exists = path.exists()
    print(f"{path}: {'OK' if exists else 'MISSING'}")
    all_ok = all_ok and exists

if not all_ok:
    raise FileNotFoundError("One or more required dataset folders are missing.")

print("Dataset structure OK")


## 7. Count files

In [ ]:
def count_files(path: Path) -> int:
    return sum(1 for p in path.rglob("*") if p.is_file())

counts = {
    "HAM10000 images": count_files(DATA_DIR / "ham10000" / "images"),
    "HAM10000 masks": count_files(DATA_DIR / "ham10000" / "masks"),
    "ISIC2018 images": count_files(DATA_DIR / "isic2018" / "images"),
    "ISIC2018 masks": count_files(DATA_DIR / "isic2018" / "masks"),
}

for name, count in counts.items():
    print(f"{name}: {count}")


## 8. Final check

In [ ]:
print("Dataset setup complete.")
print()
print("You can now run:")
print("- 01_data_inventory.ipynb")
print("- 01_data_preprocessing.ipynb")
print("- 03_create_pilot_subset.ipynb")
